# Setup

In [ ]:
# Optional autoreload while developing locally

%load_ext autoreload
%autoreload 2

In [ ]:
# to load the virtual environment
## & "$HOME\nlp_owlapy_env_py311\Scripts\Activate.ps1"

In [ ]:

# 0) Local-only setup and base diagnostics
import os
import sys
import platform
from pathlib import Path

print("Python:", sys.version)
print("Platform:", platform.platform())

PROJECT_DIR = Path.cwd()



In [ ]:

# 0.3) Shared device diagnostics
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())
    print("GPU 0:", torch.cuda.get_device_name(0))
    DEVICE = "cuda:0"
else:
    DEVICE = "cpu"
print("Using DEVICE =", DEVICE)


# Imports

In [ ]:
import pickle
import re

import pandas as pd
from spacy.tokens import Doc

from dataclasses import dataclass, asdict
from itertools import product

from tokenizer import create_tokenizer, ensure_booknlp_extensions, tokens_to_dicts
from doc_chunker import create_chunker
from runtime_config import (
    RUNTIME_PROFILE,
    DEVICE,
    CHUNK_SIZE,
    OVERLAP_SENTENCES,
    MAX_EXPANDED_CHUNK_TOKENS,
)

In [ ]:
from coreference.coreference_sub_orchestrator import (
    run_coreference_resolution,
    print_non_singleton_coref_clusters,
)

from coreference.coref_schema import require_coref_layer

In [ ]:
from ocean.ocean_probability_scoring import (
    ContextConfig,
    MentionRenderingConfig,
    OceanScoringConfig,
    OceanWeightConfig,
    DirectNLIConfig,
    OceanProbabilityExportConfig,
    export_ocean_probability_csvs,
)

from ocean.ocean_annotator import annotate_doc_with_ocean_folder
from ocean.ocean_schema import (
    require_ocean_layer,
    register_spacy_ocean_extension
)

In [ ]:
from cluster_typing.cluster_typing_probability_scoring import (
    ClusterTypingTraversalConfig,
    ClusterTypingScoringConfig,
    ClusterTypingMentionWeightConfig,
    ClusterTypingEvidenceExportConfig,
    export_cluster_typing_evidence_jsonls,
)

from cluster_typing.cluster_typing_annotator import (
    ClusterTypingAnnotationConfig,
    annotate_doc_with_cluster_typing_folder,
)

from cluster_typing.cluster_typing_schema import (
    register_spacy_cluster_typing_extension,
    require_cluster_typing_layer,
)

In [ ]:
from ontology.ontology_management import (
    assert_cluster_relation,
    assert_cluster_type,
    assert_cluster_value,
    build_relation_router,
    load_tbox,
    save_ontology,
)


In [ ]:
from relationship_extraction.relation_schema import (
    register_spacy_relation_extension,
    require_relation_layer,
)

from relationship_extraction.extract_relation_candidates import (
    extract_relation_candidates,
    export_routed_relation_candidates_jsonl,
)

from relationship_extraction.align_relation_assignments import (
    RelationNLIConfig,
    load_relation_nli_selector,
    export_relation_assignments_csv,
)

from relationship_extraction.aggregate_cluster_assertions import (
    RelationAggregationConfig,
    export_cluster_assertions_csv,
)

from relationship_extraction.annotate_relation_layer import (
    annotate_relation_layer_from_files,
)

# Functions

In [ ]:
# text loading + preprocessing
def load_txt(path):
    path = Path(path)

    try:
        return path.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        return path.read_text(encoding="latin-1")

In [ ]:
import re

def preprocess_for_NER(text):
    # Normalize Windows/Mac newlines
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Remove possible invisible BOM
    text = text.replace("\ufeff", "")

    # Normalize curly apostrophes/quotes
    text = text.replace("’", "'")
    text = text.replace("‘", "'")
    text = text.replace("“", '"').replace("”", '"')

    # Remove standalone page numbers, e.g.
    # This removes only lines that contain digits plus optional whitespace.
    text = re.sub(r"(?m)^[ \t]*\d+[ \t]*$", "", text)

    # Replace line breaks inside paragraphs with spaces,
    # but preserve paragraph boundaries
    paragraphs = text.split("\n\n")
    paragraphs = [
        re.sub(r"\s*\n\s*", " ", paragraph).strip()
        for paragraph in paragraphs
        if paragraph.strip()
    ]

    text = "\n\n".join(paragraphs)

    chapter_re = re.compile(
        r"(?im)^\s*chapter\s+(?:[ivxlcdm]+|\d+)\b[^\n]*\n*"
    )

    text = chapter_re.sub("", text)

    # Normalize multiple spaces
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()

In [ ]:

def resolve_text_path(
    *,
    project_dir: Path,
    filename: str,
    local_path: str | Path,
) -> Path:
    """Resolve the local input text path."""
    text_path = Path(local_path)
    if text_path.exists():
        return text_path

    fallback_path = project_dir / filename
    if fallback_path.exists():
        return fallback_path

    raise FileNotFoundError(
        "Could not resolve a local text file. "
        f"Tried LOCAL_TEXT_PATH={text_path} and PROJECT_DIR / {filename!r} = {fallback_path}."
    )


In [ ]:
def save_doc(doc, path: str | Path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("wb") as f:
        pickle.dump(doc, f)
    return path


def load_doc(path: str | Path):
    # BookNLP/tokenizer extensions must be registered before unpickling the Doc.
    # The global coreference extension is registered inside the coreference
    # sub-orchestrator immediately before final annotation.
    ensure_booknlp_extensions()
    register_spacy_cluster_typing_extension()
    register_spacy_ocean_extension()
    register_spacy_relation_extension()
    
    path = Path(path)
    with path.open("rb") as f:
        return pickle.load(f)


# Config

## I/O config

In [ ]:
# Requested local file
TEXT_FILENAME = "cosmicomics_a_sign_in_space.txt"
LOCAL_TEXT_PATH = Path(f"./resources/books/{TEXT_FILENAME}")

TEXT_PATH = resolve_text_path(
    project_dir=PROJECT_DIR,
    filename=TEXT_FILENAME,
    local_path=LOCAL_TEXT_PATH,
)

OUTPUT_ROOT = PROJECT_DIR / f"outputs_[{TEXT_FILENAME}]"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ONTOLOGY_ROOT = PROJECT_DIR / "resources/ontologies"

TOKENIZED_DOC_PATH = OUTPUT_ROOT / "tokenized_doc.pkl"
COREF_DOC_PATH = OUTPUT_ROOT / "coref_doc.pkl"
ONTOLOGY_TYPED_DOC_PATH = OUTPUT_ROOT / "ontology_typed_doc.pkl"
OCEAN_SCORED_DOC_PATH = OUTPUT_ROOT / "ocean_scored_doc.pkl"

ONTOLOGY_TTL_PATH = ONTOLOGY_ROOT/ "narrative_entity_ontology_clean.ttl"

RELATION_OUTPUT_DIR = OUTPUT_ROOT / "relations"
RELATION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ROUTED_RELATION_CANDIDATES_PATH = RELATION_OUTPUT_DIR / "routed_relation_candidates.jsonl"
RELATION_ASSIGNMENTS_PATH = RELATION_OUTPUT_DIR / "relation_assignments.csv"
CLUSTER_ASSERTIONS_PATH = RELATION_OUTPUT_DIR / "cluster_assertions.csv"
RELATION_DOC_PATH = OUTPUT_ROOT / "relation_doc.pkl"
POPULATED_ONTOLOGY_PATH = OUTPUT_ROOT / "populated_ontology.ttl"

print("TEXT_PATH =", TEXT_PATH)
print("OUTPUT_ROOT =", OUTPUT_ROOT)
print("TOKENIZED_DOC_PATH =", TOKENIZED_DOC_PATH)
print("COREF_DOC_PATH =", COREF_DOC_PATH)
print("ONTOLOGY_TYPED_DOC_PATH =", ONTOLOGY_TYPED_DOC_PATH)
print("OCEAN_SCORED_DOC_PATH =", OCEAN_SCORED_DOC_PATH)
print("ONTOLOGY_TTL_PATH =", ONTOLOGY_TTL_PATH)


In [ ]:
GLOBAL_COREF_OUTPUT_DIR = OUTPUT_ROOT / "global_coref"
GLOBAL_COREF_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("GLOBAL_COREF_OUTPUT_DIR =", GLOBAL_COREF_OUTPUT_DIR)

## Runtime and pipeline config

In [ ]:
# Pipeline configuration
SPACY_MODEL = "en_core_web_sm"
TOKENIZER_DISABLE = ("ner",)

RELATION_PAIR_BATCH_SIZE = 64
RELATION_OVERWRITE_STAGE_1 = False
RELATION_OVERWRITE_STAGE_2 = False
RELATION_RESUME_STAGE_2 = True
RELATION_OVERWRITE_STAGE_3 = True

print("RUNTIME_PROFILE =", RUNTIME_PROFILE)
print("DEVICE =", DEVICE)
print("CHUNK_SIZE(expanded) =", CHUNK_SIZE)
print("OVERLAP_SENTENCES =", OVERLAP_SENTENCES)
print("MAX_EXPANDED_CHUNK_TOKENS =", MAX_EXPANDED_CHUNK_TOKENS)
print("GLOBAL_COREF_OUTPUT_DIR =", GLOBAL_COREF_OUTPUT_DIR)


# Main

### Tokenization

In [ ]:
# Load or create the stable tokenized Doc artifact.
if TOKENIZED_DOC_PATH.exists():
    print(f"[doc] Loading tokenized Doc from {TOKENIZED_DOC_PATH}")
    doc = load_doc(TOKENIZED_DOC_PATH)
else:
    print("[doc] Tokenized Doc not found; creating it from raw text.")
    raw_text = load_txt(TEXT_PATH)
    text = preprocess_for_NER(raw_text)
    tokenizer = create_tokenizer(
        SPACY_MODEL,
        disable=TOKENIZER_DISABLE,
        verbose=True,
    )
    doc = tokenizer.tokenize(text)
    save_doc(doc, TOKENIZED_DOC_PATH)
    print(f"[doc] Saved tokenized Doc to {TOKENIZED_DOC_PATH}")

print("Doc tokens:", len(doc))
print("Doc sentences:", sum(1 for _ in doc.sents))

### Chunking

In [ ]:
chunker = create_chunker(
    chunk_size=CHUNK_SIZE,
    overlap_sentences=OVERLAP_SENTENCES,
    max_expanded_chunk_tokens=MAX_EXPANDED_CHUNK_TOKENS,
)
chunk_plan = chunker.plan(doc)

### Ontology building


In [ ]:
onto, class_graph = load_tbox(ONTOLOGY_TTL_PATH, require_property_descriptions=True)

print(class_graph.number_of_nodes(), "classes")
print(class_graph.number_of_edges(), "subclass edges")

relation_router = build_relation_router(
    onto=onto,
    ontology_path=ONTOLOGY_TTL_PATH,
    class_graph=class_graph,
)

print("Object properties:", len(relation_router.relation_specs))


## Node extraction

### Coreference clusters extraction

In [ ]:
if COREF_DOC_PATH.exists():
    print(f"[coref-annotation] Loading coref-annotated Doc from {COREF_DOC_PATH}")
    doc = load_doc(COREF_DOC_PATH)
else:
    print("[coref-annotation] Annotated Doc not found; calling annotator sub-orchestrator.")
    doc = run_coreference_resolution(
        doc=doc,
        chunker=chunker,
        chunk_plan=chunk_plan,
        document_id=TEXT_PATH.stem,
        output_dir=GLOBAL_COREF_OUTPUT_DIR,
    )
    save_doc(doc, COREF_DOC_PATH)
    print("[coref-annotation] Saved annotated Doc to:", COREF_DOC_PATH)
    
    
if not Doc.has_extension("coref_layer"):
    Doc.set_extension("coref_layer", default=None)

In [ ]:
print("[coref-annotation] Summary:")
print(doc._.coref_layer.summary())
print_non_singleton_coref_clusters(doc)

### Ontology cluster typing

In [ ]:
ontology_n_mentions = None  # None = type ALL mentions for each requested cluster
ontology_random_seed = 67

In [ ]:
if ONTOLOGY_TYPED_DOC_PATH.exists():
    print(f"[ontology typing] Loading ontology-typed Doc from {ONTOLOGY_TYPED_DOC_PATH}")
    doc = load_doc(ONTOLOGY_TYPED_DOC_PATH)
else:
    if not ONTOLOGY_TTL_PATH.exists():
        raise FileNotFoundError(
            f"Ontology TTL not found: {ONTOLOGY_TTL_PATH}. "
            "Create a networkx.DiGraph by any external method, or place the TTL here."
        )

    jsonl_paths_by_cluster_id = export_cluster_typing_evidence_jsonls(
        doc,
        class_graph,
        ClusterTypingEvidenceExportConfig(
            n_mentions_per_cluster=ontology_n_mentions,
            output_root=OUTPUT_ROOT,

            context_config=ContextConfig(
                n_sentences_before=0,
                n_sentences_after=0,
                mark_mention=True,
                deduplicate=False,
            ),
            rendering_config=MentionRenderingConfig(
                canonicalize_simple_mentions=True,
                keep_first_second_person=True,
            ),
            traversal_config=ClusterTypingTraversalConfig(
                skip_single_root=True,
                include_stay_option=True,
                force_leaf=False,
            ),
            scoring_config=ClusterTypingScoringConfig(
                subject_aware=True,
            ),
            mention_weight_config=ClusterTypingMentionWeightConfig(),
            nli_config=DirectNLIConfig(
                pair_batch_size=64,
            ),

            chunk_size=16,

            overwrite_jsonl=True,
            resume_from_jsonl=False,

            random_seed=ontology_random_seed,
            sort_sample_by_cluster_order=True,
            print_progress=True,
        ),
    )

    print(jsonl_paths_by_cluster_id)

    n_part = "all" if ontology_n_mentions is None else str(ontology_n_mentions)
    ontology_folder = OUTPUT_ROOT / "cluster_typing" / n_part

    doc = annotate_doc_with_cluster_typing_folder(
        doc,
        class_graph,
        ontology_folder,
        config=ClusterTypingAnnotationConfig(
            use_mention_weight=True,
        ),
    )
    save_doc(doc, ONTOLOGY_TYPED_DOC_PATH)
    print("\n\n\n[ontology typing] Saved annotated Doc to:", ONTOLOGY_TYPED_DOC_PATH)

cluster_typing_layer = require_cluster_typing_layer(doc)
print(cluster_typing_layer.summary())

In [ ]:
coref = doc._.coref_layer
cluster_typing_layer = doc._.cluster_typing_layer

rows = []

for cluster_id, cluster in sorted(coref.clusters.items()):
    rows.append(
        {
            "cluster_id": cluster_id,
            "canonical_name": cluster.canonical_name,
            "semantic_type": cluster.semantic_type,
            "n_mentions": len(cluster.mention_ids),
            "class_iri": cluster_typing_layer.class_iri(cluster_id),
            "ontology_class_label": cluster_typing_layer.class_label(cluster_id),
            "ontology_class_human_readable_label": cluster_typing_layer.class_human_readable_label(cluster_id),
        }
    )

ontology_clusters_df = pd.DataFrame(rows)

ontology_clusters_df


### OCEAN scoring

In [ ]:
def cluster_ids_of_class_or_subclass(doc, coref_layer, cluster_typing_layer, class_label):
    coref_cluster_ids = set(coref_layer.clusters)

    return [
        cluster_id
        for cluster_id in cluster_typing_layer.cluster_ids_under(class_label)
        if cluster_id in coref_cluster_ids
    ]

In [ ]:

coref = doc._.coref_layer
cluster_typing_layer = doc._.cluster_typing_layer

cluster_ids = cluster_ids_of_class_or_subclass(
    doc,
    coref,
    cluster_typing_layer,
    "Character",
)

for id in sorted(cluster_ids):
    print(id, end=', ')

In [ ]:
n_mentions = None  # None = score ALL mentions for each requested cluster
random_seed = 67

In [ ]:
if OCEAN_SCORED_DOC_PATH.exists():
    print(f"[OCEAN scoring] Loading ocean_scoring-annotated Doc from {OCEAN_SCORED_DOC_PATH}")
    doc = load_doc(OCEAN_SCORED_DOC_PATH)
else:

    csv_paths_by_cluster_id = export_ocean_probability_csvs(
    doc,
    OceanProbabilityExportConfig(
        cluster_ids=cluster_ids,
        n_mentions_per_cluster=n_mentions,
        output_root=OUTPUT_ROOT,

        context_config=ContextConfig(
            n_sentences_before=0,
            n_sentences_after=0,
            mark_mention=True,
            deduplicate=False,
        ),
        rendering_config=MentionRenderingConfig(
            canonicalize_simple_mentions=True,
            keep_first_second_person=True,
        ),
        scoring_config=OceanScoringConfig(
            subject_aware=True,
        ),
        weight_config=OceanWeightConfig(),
        nli_config=DirectNLIConfig(
            pair_batch_size=64, # on stronger hardware can be safely set to 64
        ),

        chunk_size=64,

        overwrite_csv=False,
        resume_from_csv=True,
        use_sqlite_cache=True,

        random_seed=random_seed,
        sort_sample_by_cluster_order=True,
        return_dataframes=False,
        print_progress=True,
    ),
)


    print(csv_paths_by_cluster_id)


    n_part = "all" if n_mentions is None else str(n_mentions)
    ocean_folder = OUTPUT_ROOT / "OCEAN_profiles" / n_part

    doc = annotate_doc_with_ocean_folder(
        doc,
        ocean_folder,
    )
    save_doc(doc, OCEAN_SCORED_DOC_PATH)
    print("\n\n\n[OCEAN scoring] Saved annotated Doc to:", OCEAN_SCORED_DOC_PATH)

ocean = require_ocean_layer(doc)
print(ocean.summary())
#### test

In [ ]:
coref = doc._.coref_layer
ocean = doc._.ocean_layer

for cluster_id in cluster_ids:
    print(coref.clusters[cluster_id].canonical_name, sep="->")
    print(ocean.cluster(cluster_id).scores.as_dict())

## Edge extraction

In [ ]:
for relation_iri in sorted(relation_router.relation_specs):
    print(relation_iri)


In [ ]:
def print_tree(g, root=None, indent=""):
    def label(n):
        data = g.nodes[n]
        return data.get("human_readable_label") or data.get("label") or str(n).split("#")[-1]

    if root is None:
        roots = [n for n in g.nodes if g.in_degree(n) == 0]
        for r in roots:
            print_tree(g, r)
        return

    print(indent + label(root))

    children = sorted(g.successors(root), key=label)
    for child in children:
        print_tree(g, child, indent + "    ")

print_tree(class_graph)

In [ ]:
# Inspect relation_router

import pandas as pd

def short(iri):
    iri = str(iri)
    return iri.rsplit("#", 1)[-1] if "#" in iri else iri.rstrip("/").rsplit("/", 1)[-1]

relation_specs = relation_router.relation_specs

router_df = pd.DataFrame([
    {
        "property": short(spec.iri),
        "domains": ", ".join(short(x) for x in spec.domains),
        "ranges": ", ".join(short(x) for x in spec.ranges),
        "label": spec.human_readable_label,
        "description": spec.description,
    }
    for spec in relation_specs.values()
])

print(f"Object properties in router: {len(router_df)}")
display(router_df.sort_values("property"))


In [ ]:
relation_candidates_df = extract_relation_candidates(
    doc,
    cluster_typing_layer=doc._.cluster_typing_layer,
)

relation_candidates_df[
    [
        "source_cluster_id",
        "source_canonical_name",
        "source_class_iri",
        "predicate",
        "target_cluster_id",
        "target_canonical_name",
        "target_class_iri",
        "sentence_text",
        "is_passive",
        "is_negated",
    ]
]

export_routed_relation_candidates_jsonl(
    doc=doc,
    cluster_typing_layer=doc._.cluster_typing_layer,
    relation_router=relation_router,
    output_path=ROUTED_RELATION_CANDIDATES_PATH,
    print_discards=True,
    overwrite=True,
)


In [ ]:
selector = load_relation_nli_selector(
    RelationNLIConfig(
        pair_batch_size=RELATION_PAIR_BATCH_SIZE,
    )
)

export_relation_assignments_csv(
    input_path=ROUTED_RELATION_CANDIDATES_PATH,
    output_path=RELATION_ASSIGNMENTS_PATH,
    selector=selector,
    overwrite=RELATION_OVERWRITE_STAGE_2,
    resume=RELATION_RESUME_STAGE_2,
)

In [ ]:
export_cluster_assertions_csv(
    assignments_path=RELATION_ASSIGNMENTS_PATH,
    output_path=CLUSTER_ASSERTIONS_PATH,
    aggregation_config=RelationAggregationConfig(
        aggregation_method="sum_softmax_by_cluster_pair",
        min_support_count=1,
        min_score=0.0,
    ),
    overwrite=RELATION_OVERWRITE_STAGE_3,
)

In [ ]:
relation_layer = annotate_relation_layer_from_files(
    doc=doc,
    assignments_path=RELATION_ASSIGNMENTS_PATH,
    cluster_assertions_path=CLUSTER_ASSERTIONS_PATH,
    force=True,
)

print(relation_layer.summary())

In [ ]:
coref = require_coref_layer(doc)
relations = require_relation_layer(doc)

for assignment_id in list(relations.assignments):
    source = relations.source_cluster(coref, assignment_id)
    target = relations.target_cluster(coref, assignment_id)
    predicate = relations.predicate_span(doc, assignment_id)
    prop = relations.chosen_object_property_iri(assignment_id)
    conf = relations.confidence(assignment_id)

    print(
        f"{source.canonical_name} -- {predicate.text} / {prop} "
        f"({conf:.3f}) --> {target.canonical_name}"
    )


In [ ]:
save_doc(doc, RELATION_DOC_PATH)

In [ ]:
def _local_name(iri: str) -> str:
    """Compact IRI rendering: http://x/y#livesIn -> livesIn."""
    return str(iri).rstrip("/#").rsplit("#", 1)[-1].rsplit("/", 1)[-1]


def _cluster_name(coref_layer, cluster_id: int) -> str:
    """
    Robust cluster label getter.
    Works even if your Cluster class uses slightly different field names.
    """
    cluster = coref_layer.clusters[int(cluster_id)]

    for attr in (
        "canonical_name",
        "canonical_text",
        "representative_text",
        "main_text",
        "name",
        "label",
    ):
        value = getattr(cluster, attr, None)
        if value:
            return str(value)

    # Fallback: try first mention span-like object if available.
    mentions = getattr(cluster, "mentions", None)
    if mentions:
        first = mentions[0]
        text = getattr(first, "text", None)
        if text:
            return str(text)

    return f"cluster_{cluster_id}"


relation_layer = require_relation_layer(doc)
coref_layer = require_coref_layer(doc)

relation_layer.rebuild_indexes()

assertions = sorted(
    relation_layer.cluster_assertions.items(),
    key=lambda item: (
        int(item[1].source_cluster_id),
        _local_name(item[1].object_property_iri),
        int(item[1].target_cluster_id),
    ),
)

cluster_typing_layer = doc._.cluster_typing_layer
print("=" * 100)
print(f"RELATIONSHIP CLUSTERS: {len(assertions)}")
print("=" * 100)

for i, (assertion_id, assertion) in enumerate(assertions, start=1):
    source_id = int(assertion.source_cluster_id)
    target_id = int(assertion.target_cluster_id)

    source_name = _cluster_name(coref_layer, source_id)
    target_name = _cluster_name(coref_layer, target_id)
    prop_name = _local_name(assertion.object_property_iri)

    source_type = cluster_typing_layer.class_human_readable_label(source_id)
    target_type = cluster_typing_layer.class_human_readable_label(target_id)

    support_count = relation_layer.support_count(assertion_id)
    mean_conf = relation_layer.mean_assertion_confidence(assertion_id)

    print(f"\n{source_name}({source_type}) --[{prop_name}]--> {target_name}({target_type})")